In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from pycox.models.cox import CoxPH
from pycox.evaluation import EvalSurv
import matplotlib.pyplot as plt
import torchtuples as tt
sys.path.append(os.path.abspath("../../"))
from src.dataset.generate_dataset import TorchPreprocessing
from src.dataset.DataSet import SurvivalDataSet
from src.utils.Preprocessing import Preprocessor
from src.utils.ConvertTextToCsv import TextToCsv
from src.dataset.split_data import split_data_Train_Val_Test, create_dataloaders_train_val_test
import scipy.integrate
from sklearn.preprocessing import StandardScaler
from src.utils.cox_models import *

scipy.integrate.simps = scipy.integrate.simpson
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.compare import compare_survival
from sksurv.util import Surv
from itertools import product
import torch

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
pp = Preprocessor()

In [3]:
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_data = pp.clean_columns_dataset(df_clinical_data)
list_df = pp.total_type_len_type_cancer(df_clinical_data)
df_clinical_data["Tumor-Cancer"] = list_df
df_clinical_data["Tumor-Cancer"].unique()

df_mRNA_raw_data = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")
clean_mRNA_df = pp.eliminate_zero_genes(df_mRNA_raw_data, "Hugo_Symbol")


Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 298 - Total(%) 0.36
Shape of the CSV: (20440, 819)
Genes before Treshold: 818
count     818.000000
mean     2911.273839
std       351.415192
min         0.000000
25%      2709.250000
50%      2866.500000
75%      3078.500000
max      4575.000000
dtype: float64
Threshold (>80% zeros): 16352 samples
After the treshold: 818


In [4]:
df_mrna_clean = pp.eliminate_zero_genes(df_mRNA_raw_data, "Hugo_Symbol")
df_merged = TorchPreprocessing(clean_mRNA_df,df_clinical_data, 20000).get_comparation_df()
 
comparation_df = df_merged.loc[
    df_merged["Tumor-Cancer"].isin(["Luminal A", "Luminal B", "TNBC", "HER2-enriched"]),
]

comparation_df["Tumor-Cancer"].unique()

Genes before Treshold: 818
count     818.000000
mean     2911.273839
std       351.415192
min         0.000000
25%      2709.250000
50%      2866.500000
75%      3078.500000
max      4575.000000
dtype: float64
Threshold (>80% zeros): 16352 samples
After the treshold: 818
Genes before Treshold: 20003
count    20003.000000
mean        74.215568
std        155.875304
min          0.000000
25%          0.000000
50%          0.000000
75%         21.000000
max        519.000000
dtype: float64
Threshold (>80% zeros): 415 samples
After the treshold: 18131


array(['Luminal A', 'TNBC', 'Luminal B', 'HER2-enriched'], dtype=object)

In [5]:
zero_reduced_df = pp.eliminate_zero_genes(comparation_df, "Tumor-Cancer")
print(f"Samples: {zero_reduced_df.shape[0]}, Genes: {zero_reduced_df.shape[1]}")
zero_reduced_df =  zero_reduced_df.drop(["Sample ID"], axis=1)
results_df, desing, expr = pp.initialize_limma(zero_reduced_df, column="Tumor-Cancer", column_event="Overall Survival (Months)", column_status="Overall Survival Status")

Genes before Treshold: 18131
count    18131.000000
mean        31.302907
std         83.861201
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max        415.000000
dtype: float64
Threshold (>80% zeros): 415 samples
After the treshold: 18131
Samples: 519, Genes: 18132


In [6]:
results_df

,HER2-enriched,Luminal A,Luminal B,TNBC,AveExpr,F,pvalue,adj_pvalue
YTHDF1,10.879097,10.704586,10.782180,10.716942,10.726453,69230.809848,0.000000e+00,0.000000e+00
TMEM220,5.119567,5.465502,4.987554,5.729909,5.418882,3811.390310,0.000000e+00,0.000000e+00
PTPRD,7.449424,6.828923,7.159213,6.369146,6.832669,2621.704312,0.000000e+00,0.000000e+00
LIMS3-LOC440895,4.255932,3.856995,3.848535,3.630693,3.836291,2202.210030,0.000000e+00,0.000000e+00
ZNF443,7.142934,7.330519,7.289636,6.894570,7.244427,19148.566224,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...
TMEM229A,0.349584,0.207897,0.218256,0.103054,0.198622,12.900729,1.670042e-10,1.670410e-10
PAGE2,0.285851,0.312058,0.459990,0.754457,0.406439,12.836759,1.888919e-10,1.889231e-10
SLC28A2,0.355432,0.277010,0.242954,0.086340,0.243943,12.577445,3.111162e-10,3.111505e-10
FMR1NB,0.084961,0.267415,0.215001,0.499620,0.289179,11.949239,1.040388e-09,1.040446e-09


In [7]:
N_GENES = 23500
top_genes_limma = results_df.sort_values("pvalue").index[:N_GENES]
Torch_preprocessing = TorchPreprocessing(df_mrna_clean, df_clinical_data, N_GENES)
Torch_preprocessing.genes_expression = top_genes_limma

In [8]:
Torch_preprocessing = TorchPreprocessing(df_mrna_clean, df_clinical_data, N_GENES)
Torch_preprocessing.genes_expression = top_genes_limma
X_scaled, durations, events, scaler = Torch_preprocessing.get_data_set(time_months=60)

Genes before Treshold: 18131
count    18131.000000
mean        31.302907
std         83.861201
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max        415.000000
dtype: float64
Threshold (>80% zeros): 415 samples
After the treshold: 18131


In [10]:
X_scaledÇ

NameError: name 'X_scaledÇ' is not defined